In [ ]:
# 任务4：形成月度收益与特征面板，并按论文样本规则剩除最小30%股票。

# 月收益来自本地 Monret/Monrfret；月特征来自形成月最后一个交易日。

# beta 使用当日及前五个交易日市场收益，估计窗为此前 12 个日历月。

def get_formation_panel(snap, monthly):
    first_month = pd.Period('2000-01', freq='M')
    last_month = pd.Period('2016-12', freq='M')
    is_eligible = snap['eligible'].eq(True)
    is_in_sample = snap[ym].between(first_month, last_month)
    formation_universe = snap.loc[is_eligible & is_in_sample].copy()

    return_columns = [
        stock_code, ym, monthly_return, monthly_excess_ret, 'Monrfret',
    ]
    panel = formation_universe.merge(
        monthly[return_columns],
        on=[stock_code, ym],
        how='left',
        validate='one_to_one',
        indicator='return_match',
    )
    print('已知形成股票池中缺持有月记录的行数：',int(panel['return_match'].eq('left_only').sum()))
    return panel.drop(columns='return_match').reset_index(drop=True)


monthly_all = get_formation_panel(snap, monthly)
monthly_all['size_percentile'] = np.nan
for current_month, month_data in monthly_all.groupby(ym):
    month_rank = month_data[monthly_market_value].rank(pct=True)
    monthly_all.loc[month_data.index, 'size_percentile'] = month_rank

monthly_final = monthly_all[monthly_all['size_percentile'] > 0.30]
monthly_final = monthly_final.reset_index(drop=True)

# 检查形成月与持有月相差一个月，且股票-持有月键唯一。
assert (monthly_all[ym] == monthly_all[form] + 1).all()
assert monthly_final.duplicated([stock_code, ym]).sum() == 0
print(monthly_final.shape, monthly_final[ym].nunique())

